# Brief 03 : exporter et interroger l'historique en Parquet avec duckdb

- La table `produit_historise` du brief 01 existe dans PostgreSQL. 
- Ce n'est pas la situation idéale pour au moins deux cas :
    - Si on souhaite  la partager
    - Si on souhaite l'exploiter à grande échelle

Quelques explications :
- L'accès à une base de données implique d'avoir un réseau (même si ce n'est qu'Internet)
- L'accès implique également de disposer de droits de lecture qui vont être accordés à chaque utilisateur.
- Les agrégations sur des millions d'enregistrements, même si calculée sur une seule colonne, implique de lire beaucoup de lignes de la base de données. Les calculs peuvent être lents.

Parquet et duckdb sont deux technologies qui permettent de dépasser ces limites. Un fichier autonome, en colonnes, se lit par une requête `SELECT`.

### Définitions

Parquet est un format de fichier **en colonnes**. Les valeurs d'une même colonne sont stockées et compressées ensemble. **Le volume lu diminue quand la requête ne concerne que quelques colonnes.**

duckdb est un moteur SQL analytique. Il s'exécute dans le processus (processus Python en cours d'exécution dans notre cas), sans serveur (i.e. pas besoin d'installer PostgreSQL). duckdb permet de lire un fichier Parquet directement : `SELECT ... FROM 'fichier.parquet'`.

À l'inverse de Parquet, PostgreSQL stocke une ligne d'un seul tenant. Lire une colonne implique donc de lire beaucoup de lignes. Parquet stocke une colonne "d'un seul tenant". Cette disposition sert l'agrégation d'une colonne sur beaucoup d'enregistrements. (On mesurera des temps d'exécution en Partie B).

### Compétences visées (RNCP-37638)

- C8, niveau 2 : automatiser l'extraction depuis un fichier et un système orienté analytique.
- C18, niveau 1 : identifier un format et un moteur adaptés à la volumétrie et à l'usage analytique. Première approche du data lake.
- C10, niveau 2 : produire des agrégations sur une source unique, en SQL exécuté hors SGBD.

> Prérequis data : la table `produit_historise` du brief 01, construite et contenant les données.
> 
> Prérequis installation : `pip install duckdb pyarrow`.

## Livrables

| Livrable | Forme |
|---|---|
| L'export Parquet, simple puis partitionné | script ou notebook, fichiers `.parquet` |
| Les requêtes duckdb rejouant le brief 01 | notebook, sorties identiques à PostgreSQL |
| La comparaison de taille et de temps | tableau ou cellule de sortie, deux mesures commentées |
| La note d'analyse | quelques lignes : quand préférer Parquet et duckdb, quand rester sur PostgreSQL |

## Critères d'évaluation

- les sorties duckdb sur Parquet coïncident avec celles de PostgreSQL (sur au moins trois questions )
- la mesure de temps porte sur une même question et **est commentée** (i.e. expliquée avec des commentaires)
- la note d'analyse rattache le choix d'un format à un usage

## Modalités

- Travail individuel.
- Prérequis : brief 01 (la table `produit_historise`)
- Base locale, PostgreSQL et duckdb
- Durée indicative : une demi journée

In [ ]:
# !pip install duckdb pyarrow

In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*SQLAlchemy.*")
import pandas as pd
import psycopg2
import duckdb

# ⚠️ adaptez si besoin
DATABASE_URL = "dbname=vetprice"
conn = psycopg2.connect(DATABASE_URL)
conn.autocommit = False
cur = conn.cursor()

## Partie A : premier export et premier usage de duckdb

> Le travail porte d'abord sur un fichier unique. On charge un premier dataset et on l'interroge en duckdb.

### A1. Exporter la table en Parquet

Ci dessous : 
- on crée un dataframe contenant les données de la table `produit_historise`
- puis on l'écrit en Parquet

**Comparer la taille du fichier Parquet à celle de la table côté PostgreSQL. Consigner l'écart.**

In [ ]:
ph = pd.read_sql("SELECT * FROM produit_historise", conn)
ph.to_parquet("produit_historise.parquet")

### A2. Interroger le fichier avec duckdb

Interroger le fichier directement, sans le recharger en mémoire.

In [ ]:
duckdb.sql("SELECT COUNT(*) FROM 'produit_historise.parquet'")

### A3. (FACILE) Rejouer des questions du brief 01

- Reprendre au moins trois questions déjà traitées sur `produit_historise`.
- Les réécrire en duckdb sur le fichier Parquet.

Par exemple : 
- le catalogue courant par site (`WHERE is_current`) ;
- le prix d'un produit à une date donnée (`valid_from <= D AND (valid_to IS NULL OR valid_to > D)`) ;
- l'écart de prix inter-sites pour un même `ean`.

Vérifier que les sorties soient identiques à celles obtenues en SQL sur PostgreSQL. 
Le langage reste le même, seul la source change :)

In [ ]:
# --- catalogue courant par site ---


# --- prix d'un produit à une date donnée ---


# --- écart de prix inter-sites pour un même ean ---


## Partie B : généraliser et mesurer

### B1. La dernière version par produit avec `QUALIFY`

duckdb accepte la clause `QUALIFY`. Elle filtre sur le résultat d'une fonction fenêtre, sans sous-requête. Vérifier que le résultat coïncide avec le filtre `WHERE is_current`.

In [ ]:
duckdb.sql("""
    SELECT *
    FROM 'produit_historise.parquet'
    QUALIFY ROW_NUMBER() OVER (PARTITION BY site, cle ORDER BY valid_from DESC) = 1
""")

### B2. Partitionner l'export

Exporter la table en plusieurs fichiers, un par site, dans un dossier `export/`. Interroger ensuite l'ensemble par un motif glob : `SELECT * FROM 'export/*.parquet'`. Montrer, sur une requête ne portant que sur un site, que la lecture ciblée d'un seul fichier est possible.

In [ ]:
from pathlib import Path
Path("export").mkdir(exist_ok=True)
for site, sous_df in ph.groupby("site"):
    sous_df.to_parquet(f"export/{site}.parquet")

duckdb.sql("SELECT site, COUNT(*) AS n FROM 'export/*.parquet' GROUP BY site ORDER BY n DESC")

### B3. Mesurer

Sur une même question analytique, mesurer le temps de réponse des deux côtés. Question type : le prix moyen courant par site. Rapporter les deux durées. Les commenter au regard du contraste lignes contre colonnes.

In [ ]:
# --- PostgreSQL ---


# --- duckdb sur Parquet ---
